# SageMaker Local Processing: Churn Inference

This notebook runs the `inference` container locally using SageMaker Local Mode.

Expected project layout:

```text
churn-prac/
├── pyproject.toml
├── uv.lock
└── ml/
    ├── containers/
    │   ├── inference.Dockerfile
    │   └── requirements-inference.txt
    ├── outputs/
    │   ├── processed/
    │   │   └── current.csv
    │   ├── models/
    │   │   ├── xgboost_churn_model.joblib
    │   │   └── feature_columns.json
    │   ├── predictions/
    │   └── figures/
    │       └── model/
    └── src/
        └── churn_ml/
            └── inference.py

## 1. Setup

Run this notebook from the repository root or set `PROJECT_ROOT` manually.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# If your notebook is opened from ml/notebooks, uncomment this:
PROJECT_ROOT = Path.cwd().parents[1]

ML_DIR = PROJECT_ROOT / "ml"
OUTPUTS_DIR = ML_DIR / "outputs"

PROCESSED_DIR = OUTPUTS_DIR / "processed"
MODEL_DIR = OUTPUTS_DIR / "models"

PREDICTION_DIR = OUTPUTS_DIR / "predictions"
FIGURE_DIR = OUTPUTS_DIR / "figures" / "model"

CURRENT_LOCAL = PROCESSED_DIR / "current.csv"

MODEL_LOCAL = MODEL_DIR / "xgboost_churn_model.joblib"
FEATURE_COLUMNS_LOCAL = MODEL_DIR / "feature_columns.json"

print("PROJECT_ROOT:", PROJECT_ROOT)

print("CURRENT_LOCAL exists:", CURRENT_LOCAL.exists(), CURRENT_LOCAL)

print("MODEL_LOCAL exists:", MODEL_LOCAL.exists(), MODEL_LOCAL)

print(
    "FEATURE_COLUMNS_LOCAL exists:",
    FEATURE_COLUMNS_LOCAL.exists(),
    FEATURE_COLUMNS_LOCAL,
)

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT: /Users/alex/churn-prac
CURRENT_LOCAL exists: True /Users/alex/churn-prac/ml/outputs/processed/current.csv
MODEL_LOCAL exists: True /Users/alex/churn-prac/ml/outputs/models/xgboost_churn_model.joblib
FEATURE_COLUMNS_LOCAL exists: True /Users/alex/churn-prac/ml/outputs/models/feature_columns.json


## 2. Build the local Docker image

In [2]:
!docker build -f ../containers/inference.Dockerfile -t churn-inference:local ../..


[+] Building 0.0s (0/1)                                         docker:orbstack
[+] Building 0.2s (1/2)                                         docker:orbstack
 => [internal] load build definition from inference.Dockerfile             0.0s
 => => transferring dockerfile: 330B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.2s
[+] Building 0.3s (2/2)                                         docker:orbstack
 => [internal] load build definition from inference.Dockerfile             0.0s
 => => transferring dockerfile: 330B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.3s
[+] Building 0.4s (8/10)                                        docker:orbstack
 => [internal] load build definition from inference.Dockerfile             0.0s
 => => transferring dockerfile: 330B                                       0.0s
 => [internal] load metadata for docker

## 3. Smoke test with plain Docker

This is faster to debug than SageMaker Local Mode. If this fails, fix Docker/script paths before using SageMaker.

In [3]:
!docker run --rm \
  -v "{OUTPUTS_DIR}:/opt/program/outputs" \
  churn-inference:local

{
  "n_customers_scored": 59818,
  "mean_churn_probability": 0.5458885431289673,
  "median_churn_probability": 0.5964272022247314,
  "min_churn_probability": 0.054550834000110626,
  "max_churn_probability": 0.7430338859558105,
  "n_predicted_churn_at_0_5": 39635,
  "prediction_output_path": "outputs/predictions/current_customer_churn_predictions.csv",
  "histogram_output_path": "outputs/figures/model/churn_probability_histogram.png"
}


## 4. Run with SageMaker Local Mode

This uses your already-built local image: `churn-inference:local`.

SageMaker Processing maps local inputs to `/opt/ml/processing/input/...` inside the container.

Inference outputs are written to:

- `/opt/ml/processing/output/predictions/`
- `/opt/ml/processing/output/figures/`

Input folders used in this notebook:

- `current`
- `model`

In [6]:
print(MODEL_LOCAL)
print(MODEL_LOCAL.exists())
print(MODEL_LOCAL.stat().st_size if MODEL_LOCAL.exists() else "missing")

/Users/alex/churn-prac/ml/outputs/models/xgboost_churn_model.joblib
True
171136


In [7]:
from sagemaker.local import LocalSession
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor

local_session = LocalSession()
local_session.config = {"local": {"local_code": True}}

# Local mode does not need a real AWS execution role for the local Docker run.
role = "arn:aws:iam::000000000000:role/SageMakerLocalRole"

processor = Processor(
    image_uri="churn-inference:local",
    role=role,
    instance_count=1,
    instance_type="local",
    sagemaker_session=local_session,
    entrypoint=["python", "-m", "churn_ml.inference"],
)

processor.run(
    inputs=[
        ProcessingInput(
            source=str(CURRENT_LOCAL),
            destination="/opt/ml/processing/input/current",
            input_name="current-data",
        ),
        ProcessingInput(
            source=str(MODEL_DIR),
            destination="/opt/ml/processing/model",
            input_name="model-dir",
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=str(OUTPUTS_DIR),
            output_name="inference-output",
        )
    ],
    arguments=[
        "--current-path",
        "/opt/ml/processing/input/current/current.csv",
        "--model-path",
        "/opt/ml/processing/model/xgboost_churn_model.joblib",
        "--feature-columns-path",
        "/opt/ml/processing/model/feature_columns.json",
        "--prediction-output-path",
        "/opt/ml/processing/output/predictions/current_customer_churn_predictions.csv",
        "--histogram-output-path",
        "/opt/ml/processing/output/figures/model/churn_probability_histogram.png",
    ],
)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:sagemaker:Creating processing-job with name churn-inference-2026-05-11-02-01-16-174
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-6ebve:
    container_name: 6iv45naltj-algo-1-6ebve
    entryp

time="2026-05-10T22:01:21-04:00" level=warning msg="/private/var/folders/2f/j192nttn58x0gl_t_c5x2j340000gn/T/tmpv05ff2kx/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-05-10T22:01:21-04:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpv05ff2kx\".\nSet `external: true` to use an existing network"
 Container 6iv45naltj-algo-1-6ebve  Creating
 Container 6iv45naltj-algo-1-6ebve  Created
Attaching to 6iv45naltj-algo-1-6ebve
6iv45naltj-algo-1-6ebve  | {
6iv45naltj-algo-1-6ebve  |   "n_customers_scored": 59818,
6iv45naltj-algo-1-6ebve  |   "mean_churn_probability": 0.5458885431289673,
6iv45naltj-algo-1-6ebve  |   "median_churn_probability": 0.5964272022247314,
6iv45naltj-algo-1-6ebve  |   "min_churn_probability": 0.054550834000110626,
6iv45naltj-algo-1-6ebve  |   "max_churn_probability": 0.7430338859558105,
6iv45naltj-algo-1-6ebve  |   "n_predicted_chu

INFO:sagemaker.local.image:===== Job Complete =====


## 5. Inspect outputs

In [8]:
import pandas as pd

print("=== PREDICTIONS ===")
for path in sorted(PREDICTION_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

print("\n=== FIGURES ===")
for path in sorted(FIGURE_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

predictions_path = PREDICTION_DIR / "current_customer_churn_predictions.csv"

if predictions_path.exists():
    print("\n=== TOP 10 HIGHEST CHURN RISK ===")
    predictions = pd.read_csv(predictions_path)
    display(predictions.head(10))

=== PREDICTIONS ===
current_customer_churn_predictions.csv 3370163 bytes

=== FIGURES ===
churn_probability_histogram.png 23713 bytes
feature_importance.csv 517 bytes
feature_importance.png 53835 bytes
roc_curve.png 32185 bytes

=== TOP 10 HIGHEST CHURN RISK ===


,account_id,metric_time,churn_probability,predicted_is_churn
0,dee6e861ccd3b5e21d7c90fe125c17f1,2020-05-29,0.743034,1
1,1ac02d8c0b8bde14f69d08cadc1d30d6,2020-05-29,0.743034,1
2,79d783b9f0b25c1db2a3a543a8b554b9,2020-05-29,0.743034,1
3,acee81c3442e938060d3ad82460bba4f,2020-05-29,0.740161,1
4,47d662adc654e69eb66781908e686e61,2020-05-29,0.740161,1
5,6b545639ba6d8ab4e3c6e3ee6f7a5f28,2020-05-29,0.740161,1
6,8822efafd9a13d3cabe0719f8ea5c96d,2020-05-29,0.740161,1
7,e9f9e70c6a8e7ac2af00d51bb356233d,2020-05-29,0.740161,1
8,3b4810e4ad07661f896dfa063df38d6c,2020-05-29,0.740161,1
9,c0a0fa31d1c2440881e734288417053a,2020-05-29,0.740161,1
